## Prompt Chaining

![](https://docs.dapr.io/images/dapr-agents/agents-prompt-chaining.png)

Quelle: dapr.io
---

Das Prompt-Chaining-Muster bewältigt komplexe Anforderungen, indem es Aufgaben in eine Abfolge von Schritten zerlegt, wobei jeder LLM-Aufruf die Ausgabe des vorherigen verarbeitet. Dieses Muster ermöglicht eine bessere Kontrolle des Gesamtprozesses, die Validierung zwischen den Schritten und die Spezialisierung jedes einzelnen Schrittes.

Anwendungsfälle:
* Inhaltsgenerierung (zuerst Gliederungen erstellen, dann ausarbeiten, dann überprüfen)
* Mehrstufige Analyse (Durchführung komplexer Analysen in aufeinanderfolgenden Schritten)
* Arbeitsabläufe zur Qualitätssicherung (Hinzufügen von Validierungen zwischen den Verarbeitungsschritten)


In [ ]:
from time import sleep
import dapr.ext.workflow as wf

wfr = wf.WorkflowRuntime()

@wfr.activity(name="extract_destination")
def extract_destination(ctx: wf.WorkflowActivityContext, user_input: str) -> str:
    text = (user_input or "").lower()

    if "paris" in text:
        return "Paris"

    return "Unknown"

@wfr.activity(name="create_travel_outline")
def create_travel_outline(ctx: wf.WorkflowActivityContext, destination_text: str) -> dict:
    return {
        "destination": destination_text,
        "days": 3,
        "items": [
            f"Day 1: Arrival in {destination_text}",
            f"Day 2: Main attractions in {destination_text}",
            f"Day 3: Food and departure from {destination_text}",
        ],
    }

@wfr.activity(name="expand_itinerary")
def expand_itinerary(ctx: wf.WorkflowActivityContext, travel_outline: dict) -> str:
    destination = travel_outline["destination"]
    items = travel_outline["items"]

    return "\n".join([
        f"Travel itinerary for {destination}",
        "",
        items[0] + " - Hotel check-in and evening walk",
        items[1] + " - Museum visit and city centre",
        items[2] + " - Breakfast, shopping, airport transfer",
    ])

@wfr.workflow(name="travel_planning_workflow")
def travel_planning_workflow(ctx: wf.DaprWorkflowContext, user_input: str):
    destination_text = yield ctx.call_activity(extract_destination, input=user_input)

    if "paris" not in destination_text.lower():
        return "Unable to create itinerary: Destination not recognized or supported."

    travel_outline = yield ctx.call_activity(create_travel_outline, input=destination_text)
    detailed_itinerary = yield ctx.call_activity(expand_itinerary, input=travel_outline)

    return detailed_itinerary

if __name__ == "__main__":
    wfr.start()

    try:
        client = wf.DaprWorkflowClient()

        instance_id = client.schedule_new_workflow(
            workflow=travel_planning_workflow,
            input="Plan a short trip to Paris",
        )

        print(f"Started workflow instance: {instance_id}")

        while True:
            state = client.get_workflow_state(instance_id=instance_id)
            if state.runtime_status in [wf.WorkflowStatus.COMPLETED,
                                        wf.WorkflowStatus.FAILED,
                                        wf.WorkflowStatus.TERMINATED]:
                break
            sleep(1)

        print("Workflow status:", state.runtime_status)
        print("Workflow output:", state.serialized_output)

    finally:
        wfr.shutdown()

---
### Mit KI

In [ ]:
from time import sleep
import json
import dapr.ext.workflow as wf
from dapr_agents.llm import DaprChatClient
from IPython.display import Markdown, display

wfr = wf.WorkflowRuntime()

llm = DaprChatClient(component_name="openai")


def ask_llm(prompt: str) -> str:
    messages = [
        {"role": "user", "content": prompt}
    ]

    response = llm.generate(messages=messages)

    # Rückgabe robust normalisieren
    if isinstance(response, str):
        return response.strip()

    if hasattr(response, "content"):
        return str(response.content).strip()

    # häufig: OpenAI-artige Struktur
    if hasattr(response, "choices") and response.choices:
        choice = response.choices[0]
        if hasattr(choice, "message"):
            msg = choice.message
            if hasattr(msg, "content"):
                return str(msg.content).strip()
            if isinstance(msg, dict):
                return str(msg.get("content", "")).strip()

    if isinstance(response, dict):
        choices = response.get("choices")
        if choices:
            msg = choices[0].get("message", {})
            return str(msg.get("content", "")).strip()

    return str(response).strip()


@wfr.activity(name="extract_destination")
def extract_destination(ctx: wf.WorkflowActivityContext, user_input: str) -> str:
    prompt = f"""
Extract the travel destination from the following user request.

Rules:
- Return only the destination name.
- If no destination is found, return exactly: Unknown

User request:
{user_input}
""".strip()

    result = ask_llm(prompt)
    return result or "Unknown"


@wfr.activity(name="create_travel_outline")
def create_travel_outline(ctx: wf.WorkflowActivityContext, destination_text: str) -> dict:
    prompt = f"""
Create a short travel outline for a 3-day trip to {destination_text}.

Return valid JSON only in exactly this schema:
{{
  "destination": "<destination>",
  "days": 3,
  "items": [
    "Day 1: ...",
    "Day 2: ...",
    "Day 3: ..."
  ]
}}
""".strip()

    raw = ask_llm(prompt)

    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        data = {
            "destination": destination_text,
            "days": 3,
            "items": [
                f"Day 1: Arrival in {destination_text}",
                f"Day 2: Main attractions in {destination_text}",
                f"Day 3: Food and departure from {destination_text}",
            ],
        }

    if not isinstance(data, dict):
        data = {
            "destination": destination_text,
            "days": 3,
            "items": [
                f"Day 1: Arrival in {destination_text}",
                f"Day 2: Main attractions in {destination_text}",
                f"Day 3: Food and departure from {destination_text}",
            ],
        }

    data.setdefault("destination", destination_text)
    data.setdefault("days", 3)
    data.setdefault("items", [
        f"Day 1: Arrival in {destination_text}",
        f"Day 2: Main attractions in {destination_text}",
        f"Day 3: Food and departure from {destination_text}",
    ])

    return data


@wfr.activity(name="expand_itinerary")
def expand_itinerary(ctx: wf.WorkflowActivityContext, travel_outline: dict) -> str:
    destination = travel_outline.get("destination", "the destination")

    prompt = f"""
Create a practical but concise travel itinerary for {destination}
based on this outline:

{json.dumps(travel_outline, ensure_ascii=False, indent=2)}

Requirements:
- Write in plain text
- Include morning / afternoon / evening when useful
- Keep it realistic for a short city trip
- Do not invent flights or hotel bookings
""".strip()

    result = ask_llm(prompt)

    if result:
        return result

    items = travel_outline.get("items", [])
    return "\n".join([
        f"Travel itinerary for {destination}",
        "",
        items[0] + " - Hotel check-in and evening walk" if len(items) > 0 else "Day 1 - Arrival",
        items[1] + " - Museum visit and city centre" if len(items) > 1 else "Day 2 - Sightseeing",
        items[2] + " - Breakfast, shopping, airport transfer" if len(items) > 2 else "Day 3 - Departure",
    ])


@wfr.workflow(name="travel_planning_workflow")
def travel_planning_workflow(ctx: wf.DaprWorkflowContext, user_input: str):
    destination_text = yield ctx.call_activity(extract_destination, input=user_input)

    if destination_text.lower() == "unknown":
        return "Unable to create itinerary: Destination not recognized."

    travel_outline = yield ctx.call_activity(create_travel_outline, input=destination_text)
    detailed_itinerary = yield ctx.call_activity(expand_itinerary, input=travel_outline)

    return detailed_itinerary

if __name__ == "__main__":
    wfr.start()

    try:
        client = wf.DaprWorkflowClient()

        instance_id = client.schedule_new_workflow(
            workflow=travel_planning_workflow,
            input="Plan a short trip to Paris with good food and museums",
        )

        print(f"Started workflow instance: {instance_id}")

        while True:
            state = client.get_workflow_state(instance_id=instance_id)
            if state.runtime_status in [
                wf.WorkflowStatus.COMPLETED,
                wf.WorkflowStatus.FAILED,
                wf.WorkflowStatus.TERMINATED,
            ]:
                break
            sleep(1)

        print("Workflow status:", state.runtime_status)
        print("Workflow output:", state.serialized_output)
   

    finally:
        wfr.shutdown()